**Churn Modelling**

In [264]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder, OneHotEncoder
import pickle

In [265]:
data = pd.read_csv('Churn_Modelling.csv')

In [266]:
data.head()

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


**Preprocess the Data**

In [267]:
# Drop Irrelevant Columns

data = data.drop(['RowNumber', 'CustomerId', 'Surname'], axis=1)

In [268]:
data.head()

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [269]:
# Encode Categorical Variable
# LabelEncoder for Gender -> Good
# One Hot Encoding for Geography -> Good

gender_encoder = LabelEncoder()
geography_encoder = OneHotEncoder(sparse_output=False)

data['Gender'] = gender_encoder.fit_transform(data['Gender'])
geo_encoder = geography_encoder.fit_transform(data[['Geography']])

In [270]:
geography_encoder.get_feature_names_out(['Geography'])

array(['Geography_France', 'Geography_Germany', 'Geography_Spain'],
      dtype=object)

In [271]:
geo_encoded_df = pd.DataFrame(geo_encoder, columns=geography_encoder.get_feature_names_out(['Geography']))

In [272]:
geo_encoded_df

,Geography_France,Geography_Germany,Geography_Spain
0,1.0,0.0,0.0
1,0.0,0.0,1.0
2,1.0,0.0,0.0
3,1.0,0.0,0.0
4,0.0,0.0,1.0
...,...,...,...
9995,1.0,0.0,0.0
9996,1.0,0.0,0.0
9997,1.0,0.0,0.0
9998,0.0,1.0,0.0


In [273]:
data = pd.concat([data.drop('Geography', axis=1),geo_encoded_df], axis=1)
data.head()

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Geography_France,Geography_Germany,Geography_Spain
0,619,0,42,2,0.00,1,1,1,101348.88,1,1.0,0.0,0.0
1,608,0,41,1,83807.86,1,0,1,112542.58,0,0.0,0.0,1.0
2,502,0,42,8,159660.80,3,1,0,113931.57,1,1.0,0.0,0.0
3,699,0,39,1,0.00,2,0,0,93826.63,0,1.0,0.0,0.0
4,850,0,43,2,125510.82,1,1,1,79084.10,0,0.0,0.0,1.0


In [274]:
# save the encoders and scalers

with open('label_encoder_gender.pkl', 'wb') as file:
    pickle.dump(gender_encoder, file)

with open('one_hot_encoder_geography.pkl', 'wb') as file:
    pickle.dump(geography_encoder, file)

In [275]:
# Divide the Dataset into dependent and independent features and Scaling

X = data.drop('Exited', axis=1)
y = data['Exited']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [276]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.fit_transform(X_test)

In [277]:
with open('scaler.pkl', 'wb') as file:
    pickle.dump(scaler, file)

**ANN Implementation**

In [278]:
import tensorflow as tf

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.callbacks import EarlyStopping, TensorBoard
import datetime

In [279]:
##Build our ANN model

model = Sequential([
    Dense(64, activation='relu', input_shape = (X_train.shape[1],)), #Hidden Layer - 1 (Connected with Input Layer)
    Dense(32, activation='relu'), # Hidden Layer 2
    Dense(1, activation='sigmoid')
])

c:\Users\mayan\AppData\Local\Programs\Python\Python310\lib\site-packages\keras\src\layers\core\dense.py:95: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [280]:
model.summary()

Model: "sequential_9"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_25 (Dense)                │ (None, 64)             │           832 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_26 (Dense)                │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_27 (Dense)                │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,945 (11.50 KB)

 Trainable params: 2,945 (11.50 KB)

 Non-trainable params: 0 (0.00 B)

In [281]:
opt = tf.keras.optimizers.Adam(learning_rate=0.01)
loss = tf.keras.losses.BinaryCrossentropy()

In [282]:
# Compile the model

model.compile(optimizer=opt, loss='binary_crossentropy', metrics=['accuracy'])

In [283]:
# Setup the TensorBoard

log_dir = "logs/fit/" + datetime.datetime.now().strftime("%Y%m%d-%H%M%S")

In [284]:
tenserflow_callback = TensorBoard(log_dir=log_dir, histogram_freq=1)

In [285]:
#Setup EarlyStopping
earlystoping_callback = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

In [286]:
## Train the Model

history = model.fit(
    X_train, y_train, validation_data = (X_test, y_test), epochs= 100,
    callbacks = [tenserflow_callback, earlystoping_callback]
)

Epoch 1/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.8374 - loss: 0.3927 - val_accuracy: 0.8620 - val_loss: 0.3521
Epoch 2/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.8541 - loss: 0.3569 - val_accuracy: 0.8565 - val_loss: 0.3493
Epoch 3/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.8585 - loss: 0.3487 - val_accuracy: 0.8485 - val_loss: 0.3771
Epoch 4/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.8580 - loss: 0.3428 - val_accuracy: 0.8575 - val_loss: 0.3465
Epoch 5/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.8655 - loss: 0.3412 - val_accuracy: 0.8555 - val_loss: 0.3615
Epoch 6/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.8612 - loss: 0.3401 - val_accuracy: 0.8585 - val_loss: 0.3405
Epoch 7/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 3s 5ms/step - accuracy: 0.8631 - loss: 0.3371 - val_accuracy: 0.8600 - val_loss: 0.3385
Epoch 8/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 3s 6ms/step - accuracy: 0.8664 - loss: 0.3317 - val_accu

In [287]:
model.save('model.h5')

In [288]:
# Load TensorBoard Extention

%load_ext tensorboard

The tensorboard extension is already loaded. To reload it, use:
  %reload_ext tensorboard


In [289]:
%tensorboard --logdir logs/fit/

Reusing TensorBoard on port 6010 (pid 976), started 0:11:21 ago. (Use '!kill 976' to kill it.)

In [ ]:
## Load the pickel file

